# 03 — Region proposal por HSV

Probamos el módulo `src/region_proposal` sobre imágenes reales del dataset combinado.

**Objetivo:** ver visualmente las cajas candidatas, entender qué hace cada paso (máscaras, morfología, NMS) y experimentar con los parámetros.

**Importante:** el region proposal **no tiene que ser perfecto**. Su objetivo es alto recall (detectar todos los productos) aunque haya falsos positivos. El clasificador SVM filtrará los falsos en F3.

In [ ]:
import sys
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import preprocess
from src.region_proposal import (
    propose_regions, draw_proposals,
    build_mask, clean_mask, boxes_from_mask,
    HSV_RANGES, LOW_SATURATION_RANGE,
)
from src.utils.io_utils import load_image, list_images

DATA_ROOT = Path('../data/external/combined')
plt.rcParams['figure.dpi'] = 90

## 1. Cargar una imagen de prueba

Para empezar elegimos un producto con color saturado (manzana). Si quieres probar con otra cosa, cambia `category` a cualquier carpeta de `data/external/combined/` (Apple, CEREAL, JUICE, JAM, CHIPS, COFFEE, TEA…).

In [ ]:
category = 'Apple'  # cambia para probar con otros productos
images = list_images(DATA_ROOT / category)

if not images:
    print(f'⚠  No hay imágenes en {category}.')
else:
    img = load_image(images[0])
    img_pre = preprocess(img)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(img_pre); axes[1].set_title('Preprocesada (WB+CLAHE+Denoise)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

## 2. Pipeline completo de region proposal

Llamamos a `propose_regions()` con los parámetros por defecto del proyecto.

In [ ]:
proposals = propose_regions(img_pre)
print(f'Generadas {len(proposals)} propuestas')
print()
print('Top 5 por área:')
for p in sorted(proposals, key=lambda p: -p.area)[:5]:
    print(f'  {p.category:10s}  box=({p.x},{p.y},{p.w},{p.h})  area={p.area}  ar={p.aspect_ratio:.2f}')

annotated = draw_proposals(img_pre, proposals)
plt.figure(figsize=(9, 6))
plt.imshow(annotated)
plt.title(f'{len(proposals)} propuestas')
plt.axis('off')
plt.tight_layout(); plt.show()

## 3. Desglose por etapas

Vamos viendo lo que pasa por dentro: para cada categoría de color, máscara cruda → máscara limpia → overlay sobre la imagen.

In [ ]:
bgr = cv2.cvtColor(img_pre, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

categories_to_show = list(HSV_RANGES.keys())
n = len(categories_to_show)

fig, axes = plt.subplots(n, 3, figsize=(11, 2.4 * n))

for i, cat in enumerate(categories_to_show):
    ranges = HSV_RANGES[cat]
    raw_mask = build_mask(hsv, ranges)
    clean = clean_mask(raw_mask)
    overlay = img_pre.copy()
    overlay[clean == 0] = (overlay[clean == 0] * 0.3).astype(np.uint8)

    axes[i, 0].imshow(raw_mask, cmap='gray'); axes[i, 0].set_title(f'{cat} — máscara cruda', fontsize=9)
    axes[i, 1].imshow(clean, cmap='gray');   axes[i, 1].set_title(f'{cat} — limpia', fontsize=9)
    axes[i, 2].imshow(overlay);              axes[i, 2].set_title(f'{cat} — sobre la imagen', fontsize=9)
    for ax in axes[i]: ax.axis('off')

plt.tight_layout(); plt.show()

## 4. Galería sobre varias categorías del dataset

Categorías que existen en el dataset combinado y son representativas.

In [ ]:
categories_test = ['Apple', 'Banana', 'CEREAL', 'JUICE', 'CHIPS', 'JAM']

fig, axes = plt.subplots(len(categories_test), 2, figsize=(9, 3.5 * len(categories_test)))
for i, cat in enumerate(categories_test):
    cat_dir = DATA_ROOT / cat
    if not cat_dir.is_dir():
        continue
    imgs = list_images(cat_dir)
    if not imgs:
        continue
    img = load_image(imgs[0])
    img_pre = preprocess(img)
    proposals = propose_regions(img_pre)
    annotated = draw_proposals(img_pre, proposals)

    axes[i, 0].imshow(img_pre); axes[i, 0].set_title(f'{cat} — preprocesada', fontsize=10); axes[i, 0].axis('off')
    axes[i, 1].imshow(annotated); axes[i, 1].set_title(f'{cat} — {len(proposals)} cajas', fontsize=10); axes[i, 1].axis('off')

plt.tight_layout(); plt.show()

## 5. Sandbox: ajusta parámetros

Estos son los **parámetros actualizados del proyecto**, ajustados tras experimentación. Puedes cambiarlos para ver cómo afectan.

In [ ]:
img = load_image(list_images(DATA_ROOT / 'Apple')[0])
img_pre = preprocess(img)

# Parámetros actuales del proyecto (sintonizados tras pruebas)
params = dict(
    kernel_size=7,           # tamaño del kernel morfológico
    closing_iters=2,         # cierra agujeros pequeños
    opening_iters=2,         # elimina ruido fino
    min_area_ratio=0.005,    # descarta cajas microscópicas (< 0.5% del área)
    max_area_ratio=0.5,      # descarta cajas que cubren toda la imagen
    min_aspect=0.2,          # ancho/alto mínimo
    max_aspect=5.0,          # ancho/alto máximo
    iou_threshold=0.3,       # NMS más agresivo (umbral más bajo)
    use_low_saturation=True, # incluir categoría 'neutro' para blancos/grises
)

proposals = propose_regions(img_pre, **params)
annotated = draw_proposals(img_pre, proposals)

plt.figure(figsize=(9, 6))
plt.imshow(annotated)
plt.title(f'{len(proposals)} propuestas con parámetros actuales')
plt.axis('off')
plt.tight_layout(); plt.show()

print('Resumen por categoría de color:')
for cat, n in Counter(p.category for p in proposals).items():
    print(f'  {cat:10s} {n}')

## 6. Cosas a observar

Los parámetros que ves arriba son los que dieron mejor balance entre recall y limpieza tras experimentación. Si los relajas verás:

- **Bajar `kernel_size` a 5**: más ruido (motas pequeñas no se eliminan).
- **Bajar `opening_iters` a 1**: más cajas duplicadas alrededor del mismo objeto.
- **Bajar `min_area_ratio` a 0.001**: cajas microscópicas que no son productos.
- **Subir `iou_threshold` a 0.4-0.5**: menos NMS, más cajas redundantes.

**Próximo paso:** `04_evaluacion_svm.ipynb` para ver cómo se clasifican las cajas con el SVM entrenado.